In [15]:
import joblib
import numpy as np
import pandas as pd

!pip install catboost xgboost lightgbm -q

pipeline       = joblib.load("/content/drive/MyDrive/Model/gwq_full_stack.pkl")
model_xgb      = pipeline["xgb"]
model_cat      = pipeline["cat"]
model_lgb      = pipeline["lgb"]
model_rf       = pipeline["rf"]
meta_model     = pipeline["meta_model"]
scaler         = pipeline["scaler"]
imputer        = pipeline["imputer"]
num_features   = pipeline["num_features"]
all_features   = pipeline["all_features"]

def predict_wqi(ammonia, bod, dissolved_oxygen, orthophosphate,
                ph, temperature, nitrogen, nitrate,
                country=0, area=0, waterbody_type=0):
    sample_df = pd.DataFrame([[
        ammonia, bod, dissolved_oxygen, orthophosphate,
        ph, temperature, nitrogen, nitrate,
        country, area, waterbody_type
    ]], columns=all_features)
    sample_df[num_features] = imputer.transform(sample_df[num_features])
    p1 = model_xgb.predict(sample_df)[0]
    p2 = model_cat.predict(sample_df)[0]
    p3 = model_lgb.predict(sample_df)[0]
    p4 = model_rf.predict(sample_df)[0]
    meta_input = scaler.transform([[p1, p2, p3, p4]])
    return float(meta_model.predict(meta_input)[0])

def classify_wqi(wqi):
    if wqi >= 95:   return "Excellent"
    elif wqi >= 80: return "Good"
    elif wqi >= 65: return "Fair"
    elif wqi >= 45: return "Marginal"
    else:           return "Poor"


# All 6 available sensors (Ammonia and Orthophosphate offline)
wqi_full = predict_wqi(
    ammonia=0.06,        # median from Table 2
    bod=2.70,            # median from Table 2
    dissolved_oxygen=10.20,  # median from Table 2
    orthophosphate=0.11, # median from Table 2
    ph=7.78,             # median from Table 2
    temperature=11.46,   # median from Table 2
    nitrogen=0.13,       # mean (not reported in Table 2)
    nitrate=4.50         # median from Table 2
)

# Ammonia + Orthophosphate sensors fail simultaneously
wqi_degraded = predict_wqi(
    ammonia=np.nan,      # sensor failure
    bod=2.70,
    dissolved_oxygen=10.20,
    orthophosphate=np.nan,  # sensor failure
    ph=7.78,
    temperature=11.46,
    nitrogen=0.13,
    nitrate=4.50
)

print("=" * 55)
print("  GWQ-Stack Fault Tolerance Validation")
print("  Scenario: Ammonia + Orthophosphate failure")
print("=" * 55)
print(f"  All sensors present  : {wqi_full:.4f}  [{classify_wqi(wqi_full)}]")
print(f"  Dual sensor failure  : {wqi_degraded:.4f}  [{classify_wqi(wqi_degraded)}]")
print(f"  Deviation            : {abs(wqi_full - wqi_degraded):.4f} WQI units")
print(f"  Quality class change : {'YES' if classify_wqi(wqi_full) != classify_wqi(wqi_degraded) else 'None'}")
print("=" * 55)

  GWQ-Stack Fault Tolerance Validation
  Scenario: Ammonia + Orthophosphate failure
  All sensors present  : 91.3635  [Good]
  Dual sensor failure  : 91.4387  [Good]
  Deviation            : 0.0752 WQI units
  Quality class change : None
